In [ ]:
# Model 2 with LoRA
# Overlapping chunks and label-wise pooling
# Settings and helpers
import os, re, sys, gc, json, math, time, random, shutil, zipfile, contextlib
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

# Set True for a quick test run.
# Set it back to False for full training.
SMOKE_TEST = False

RUN_TAG = "p5_lsi_m2_lora_v1" + ("_smoke" if SMOKE_TEST else "")
PHASE1_MODEL_TAG = "decoder_only_rope_gqa_swiglu_clean_bpe32k_v1"   # tag written by the Model-2 notebook

IS_KAGGLE = os.path.isdir("/kaggle/working")
INPUT_ROOT = "/kaggle/input"
WORK_DIR = ("/kaggle/working" if IS_KAGGLE else ".") + f"/{RUN_TAG}"
CKPT_DIR = f"{WORK_DIR}/checkpoints"
CACHE_DIR = f"{WORK_DIR}/cache"
OUT_DIR = f"{WORK_DIR}/submission"
for _d in (CKPT_DIR, CACHE_DIR, OUT_DIR):
    os.makedirs(_d, exist_ok=True)

# Optional manual overrides. Leave "" to auto-search /kaggle/input.
TOKENIZER_DIR = ""      # folder that contains tokenizer.json (BPE notebook output)
PHASE1_CKPT = ""        # full path of the Model-2 best.pt (Phase-1 pretraining output)

# Data
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
PHASE5_FILES = {
    "train": "Phase_5_LSI/Train/train.json",
    "dev":   "Phase_5_LSI/Train/dev.json",
    "test":  "Phase_5_LSI/Test/test.json",
}

# Tokenization
SIMPLIFY_PLACEHOLDERS = True      # "<ENTITY>" -> "ENTITY": saves 2 BPE tokens per placeholder (anonymised corpus)
MAX_CHARS_HEAD = 50_000           # docs longer than head+tail chars keep their first 50k + last 30k chars
MAX_CHARS_TAIL = 30_000           # (only ~0.5% of docs; p99 = 48.7k chars)

# Chunking
CHUNK_LEN = 1024
STRIDE = 768                      # 256-token overlap between consecutive chunks
OVERLAP = CHUNK_LEN - STRIDE
MAX_CHUNKS_TRAIN = 5              # random chunk subset per doc in training (first + last always kept)
MAX_CHUNKS_EVAL = 16              # deterministic evenly-spaced chunks at inference (covers ~12k tokens fully)

# LoRA settings
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
USE_RSLORA = False                # True -> scale = alpha / sqrt(r)
LORA_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "w_gate", "w_up", "w_down")
FT_DROPOUT = 0.1                  # residual dropout inside the frozen backbone during fine-tuning (pretrain: 0.2)

# Classification head
HEAD_HIDDEN = 512
HEAD_DROPOUT = 0.1
POS_WEIGHT_CAP = 5.0              # pos_weight = clip(sqrt(neg/pos), 1, cap)

# Training settings
EPOCHS = 3
TRAIN_TOKEN_BUDGET = 24 * CHUNK_LEN   # max (chunks x chunk_len) per optimizer step
EVAL_TOKEN_BUDGET = 64 * CHUNK_LEN
MAX_DOCS_PER_BATCH = 48
DOC_NORM = 16.0                       # loss = sum over docs / DOC_NORM  (every doc weighs the same)
LORA_LR = 4e-4
HEAD_LR = 1.5e-3
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 200
MIN_LR_RATIO = 0.05
GRAD_CLIP = 1.0
GRAD_CKPT = True                      # gradient checkpointing (less memory, ~30% slower). Set False if you have memory to spare.
SAVE_EVERY = 400                      # optimizer steps
LOG_EVERY = 50
TIME_BUDGET_H = 9.5                   # whole session; lower it if your free-tier quota is smaller
FINAL_RESERVE_H = 0.5                 # kept free for dev tuning + test prediction
SEED = 1234

if SMOKE_TEST:
    N_SMOKE = {"train": 600, "dev": 300, "test": 200}
    EPOCHS, SAVE_EVERY, LOG_EVERY, WARMUP_STEPS = 1, 10, 5, 10
else:
    N_SMOKE = None


# Helper functions
def log(*a):
    print(*a, flush=True)


def fmt_time(sec):
    sec = int(max(sec, 0))
    return f"{sec // 3600}h{(sec % 3600) // 60:02d}m"


def read_json(path):
    try:
        with open(path) as f:
            return json.load(f)
    except Exception:
        return None


def atomic_json(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=1)
    os.replace(tmp, path)


def atomic_torch_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)


def walk_find(filename, roots):
    hits = []
    for root in roots:
        if not os.path.isdir(root):
            continue
        for d, _, files in os.walk(root):
            if filename in files:
                hits.append(d)
    return hits


def get_hf_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF")      # Kaggle secret named HF
    except Exception:
        return None


random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = {"enabled": False, "dtype": torch.float16}
if DEVICE.type == "cuda":
    AMP["enabled"] = True
    AMP["dtype"] = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
    torch.backends.cuda.matmul.allow_tf32 = True
    log(f"GPU: {torch.cuda.get_device_name()} | autocast dtype: {AMP['dtype']}")
else:
    log("WARNING: no GPU visible - this will be extremely slow")


def amp_ctx():
    return torch.autocast(device_type="cuda", dtype=AMP["dtype"], enabled=AMP["enabled"])


# Find the pretrained Model 2 checkpoint.
def find_phase1_ckpt():
    if PHASE1_CKPT:
        return PHASE1_CKPT
    cands = []
    for d, _, files in os.walk(INPUT_ROOT):
        if "best.pt" in files:
            meta = read_json(os.path.join(d, "best_meta.json")) or {}
            ok = meta.get("tag") == PHASE1_MODEL_TAG
            cands.append((0 if ok else 1, meta.get("val_loss", 1e9), os.path.join(d, "best.pt")))
    if not cands:
        for d, _, files in os.walk(INPUT_ROOT):
            if "latest.pt" in files:
                meta = read_json(os.path.join(d, "latest_meta.json")) or {}
                ok = meta.get("tag") == PHASE1_MODEL_TAG
                cands.append((0 if ok else 1, -meta.get("global_step", 0), os.path.join(d, "latest.pt")))
    if not cands:
        raise FileNotFoundError(
            "No Model-2 best.pt / latest.pt under /kaggle/input. Attach the Phase-1 Model-2 output as a dataset "
            "or set PHASE1_CKPT manually.")
    cands.sort()
    if cands[0][0] == 1:
        log("WARNING: no checkpoint with the Model-2 tag found; using", cands[0][2])
    return cands[0][2]


# Load the tokenizer, or rebuild it from vocab and merges.
class LegalByteBPE:
    def __init__(self, tk):
        self.tokenizer = tk
        self.vocab_size = tk.get_vocab_size()
        v = tk.get_vocab()
        self.pad_id, self.unk_id, self.eos_id = v["<|pad|>"], v["<|unk|>"], v["<|endoftext|>"]


def _tokenizer_from_vocab_merges(d):
    from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, decoders
    with open(os.path.join(d, "vocab.json"), encoding="utf-8") as f:
        vocab = json.load(f)
    with open(os.path.join(d, "merges.json"), encoding="utf-8") as f:
        merges = [tuple(m) if isinstance(m, list) else tuple(m.split(" ", 1)) for m in json.load(f)]
    tk = Tokenizer(models.BPE(vocab=vocab, merges=merges, unk_token="<|unk|>"))
    tk.normalizer = normalizers.NFKC()
    tk.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tk.decoder = decoders.ByteLevel()
    return tk


def load_tokenizer(expected_vocab):
    try:
        from tokenizers import Tokenizer
    except ImportError:
        os.system("pip -q install tokenizers")
        from tokenizers import Tokenizer
    roots = [TOKENIZER_DIR] if TOKENIZER_DIR else [INPUT_ROOT]
    for root in roots:
        for d, _, files in os.walk(root):
            if "tokenizer.json" in files:
                tk = Tokenizer.from_file(os.path.join(d, "tokenizer.json"))
                if tk.get_vocab_size() == expected_vocab:
                    log("tokenizer.json found in", d)
                    return LegalByteBPE(tk)
    for root in roots:
        for d, _, files in os.walk(root):
            if "vocab.json" in files and "merges.json" in files:
                tk = _tokenizer_from_vocab_merges(d)
                if tk.get_vocab_size() == expected_vocab:
                    log("tokenizer rebuilt from vocab.json + merges.json in", d)
                    return LegalByteBPE(tk)
    raise FileNotFoundError(f"No tokenizer with vocab_size={expected_vocab} found (attach the BPE notebook output).")


log("config ready | run tag:", RUN_TAG, "| smoke:", SMOKE_TEST)

In [ ]:
# Load checkpoint, tokenizer, data, labels, and token cache
from huggingface_hub import hf_hub_download, login

# Load the pretrained checkpoint on CPU.
CKPT_PATH = find_phase1_ckpt()
log("Phase-1 checkpoint:", CKPT_PATH)
PH1 = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
PH1_CFG = PH1.get("config") or {}
EXPECTED_VOCAB = int(PH1_CFG.get("vocab_size", 32000))
log("Phase-1 config:", PH1_CFG, "| val_loss:", PH1.get("val_loss"))

# Tokenizer
tok = load_tokenizer(EXPECTED_VOCAB)
PAD_ID = tok.pad_id
log(f"vocab={tok.vocab_size} pad={tok.pad_id} unk={tok.unk_id} eos={tok.eos_id}")

# Load training data
HF_TOKEN = get_hf_token()
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN)


def load_records(split):
    path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=PHASE5_FILES[split],
                           repo_type="dataset", token=HF_TOKEN)
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    if isinstance(obj, dict):
        for key in ("data", "records", "examples"):
            if isinstance(obj.get(key), list):
                obj = obj[key]
                break
    assert isinstance(obj, list), f"unexpected JSON structure in {split}"
    log(f"{split:5s}: {len(obj):,} documents")
    return obj


raw = {s: load_records(s) for s in ("train", "dev", "test")}


def parse_labels(lst):
    out = set()
    for x in lst:
        if isinstance(x, bool):
            continue
        if isinstance(x, (int, np.integer)):
            out.add(int(x))
        elif isinstance(x, str) and x.strip().isdigit():
            out.add(int(x.strip()))
    return sorted(out)          # sorted + unique  (3 train docs / 1 dev doc contain duplicate labels)


# Use all labels found in train or dev.
LABEL_IDS = sorted({l for s in ("train", "dev") for r in raw[s] for l in parse_labels(r["labels"])})
LAB2IDX = {l: i for i, l in enumerate(LABEL_IDS)}
N_LABELS = len(LABEL_IDS)
log(f"{N_LABELS} statutes | id range {LABEL_IDS[0]}..{LABEL_IDS[-1]}")

if SMOKE_TEST:                  # tiny deterministic subsets (labels were computed on the full data above)
    rs = random.Random(SEED)
    for s in raw:
        keep = sorted(rs.sample(range(len(raw[s])), min(N_SMOKE[s], len(raw[s]))))
        raw[s] = [raw[s][i] for i in keep]
        log(f"SMOKE: {s} -> {len(raw[s])} docs")


def build_Y(recs):
    Y = np.zeros((len(recs), N_LABELS), dtype=np.uint8)
    for i, r in enumerate(recs):
        for l in parse_labels(r["labels"]):
            Y[i, LAB2IDX[l]] = 1
    return Y


Y_train, Y_dev = build_Y(raw["train"]), build_Y(raw["dev"])
log(f"labels/doc: train {Y_train.sum(1).mean():.2f} | dev {Y_dev.sum(1).mean():.2f} | "
    f"min/max positives per label (train): {Y_train.sum(0).min()}/{Y_train.sum(0).max()}")

# Text preprocessing
_PH = re.compile(r"<([A-Z][A-Z_]*)>")
_WS = re.compile(r"\s+")


def prep_text(t):
    if not isinstance(t, str):
        t = ""
    if len(t) > MAX_CHARS_HEAD + MAX_CHARS_TAIL:
        t = t[:MAX_CHARS_HEAD] + " " + t[-MAX_CHARS_TAIL:]       # keep beginning (facts/charges) and end (holding)
    if SIMPLIFY_PLACEHOLDERS:
        t = _PH.sub(r"\1", t)
    return _WS.sub(" ", t).strip()


class TokStore:
    """all documents of one split as one flat uint16 array + offsets"""

    def __init__(self, tokens, offsets):
        self.tokens = tokens
        self.offsets = offsets
        self.lens = np.diff(offsets)

    def __len__(self):
        return len(self.lens)


def tokenize_split(name, recs):
    sig = {"n": len(recs), "vocab": tok.vocab_size, "simplify": SIMPLIFY_PLACEHOLDERS,
           "head": MAX_CHARS_HEAD, "tail": MAX_CHARS_TAIL,
           "first": str(recs[0]["id"]), "last": str(recs[-1]["id"])}
    npz, sigp = f"{CACHE_DIR}/{name}_tokens.npz", f"{CACHE_DIR}/{name}_sig.json"
    if os.path.exists(npz) and read_json(sigp) == sig:
        z = np.load(npz)
        log(f"{name}: tokens loaded from cache")
        return TokStore(z["tokens"], z["offsets"])
    t0 = time.time()
    chunks, lens = [], []
    BS = 256
    for i in range(0, len(recs), BS):
        texts = [prep_text(r["text"]) for r in recs[i:i + BS]]
        for e in tok.tokenizer.encode_batch(texts):
            ids = e.ids if len(e.ids) > 0 else [tok.eos_id]
            chunks.append(np.asarray(ids, dtype=np.uint16))
            lens.append(len(ids))
        if (i // BS) % 20 == 0:
            log(f"  {name}: {i + len(texts):,}/{len(recs):,} docs | {(time.time() - t0) / 60:.1f} min")
    offsets = np.zeros(len(lens) + 1, dtype=np.int64)
    offsets[1:] = np.cumsum(lens)
    tokens = np.concatenate(chunks)
    np.savez(npz, tokens=tokens, offsets=offsets)
    atomic_json(sig, sigp)
    return TokStore(tokens, offsets)


stores = {s: tokenize_split(s, raw[s]) for s in ("train", "dev", "test")}
for s, st in stores.items():
    q = np.percentile(st.lens, [50, 90, 95, 99])
    log(f"{s:5s}: {len(st):,} docs | tokens/doc median {q[0]:.0f} p90 {q[1]:.0f} p95 {q[2]:.0f} p99 {q[3]:.0f} "
        f"max {st.lens.max():,} | total {st.lens.sum() / 1e6:.1f}M tokens")

TEST_IDS = [r["id"] for r in raw["test"]]          # original ids, original order
for s in raw:                                      # raw texts are no longer needed
    for r in raw[s]:
        r.pop("text", None)
gc.collect()

In [ ]:
# Chunking, metrics, and threshold tuning

# Chunk geometry
def n_chunks_of(n):
    """number of CHUNK_LEN windows (stride STRIDE) needed to cover n tokens"""
    return -(-max(1, n - OVERLAP) // STRIDE)


def chunk_starts(n):
    return list(range(0, max(1, n - OVERLAP), STRIDE))


def pick_starts(starts, k, rng=None):
    """<=k chunk starts. rng=None: evenly spaced (deterministic). rng given: first + last + random middle."""
    n = len(starts)
    if n <= k:
        return starts
    if rng is None:
        idx = np.unique(np.round(np.linspace(0, n - 1, k)).astype(int))
    else:
        mid = rng.choice(np.arange(1, n - 1), size=k - 2, replace=False) if n > 2 and k > 2 else np.array([], dtype=int)
        idx = np.sort(np.concatenate([[0], mid, [n - 1]]).astype(int))
    return [starts[i] for i in idx]


# Metrics
def f1_per_label(Y, pred):
    Y, pred = Y.astype(bool), pred.astype(bool)
    tp = (pred & Y).sum(0).astype(np.float64)
    fp = (pred & ~Y).sum(0).astype(np.float64)
    fn = (~pred & Y).sum(0).astype(np.float64)
    den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)


def all_metrics(Y, pred):
    Y, pred = Y.astype(bool), pred.astype(bool)
    tp, fp, fn = (pred & Y).sum(), (pred & ~Y).sum(), (~pred & Y).sum()
    return {"macro_f1": float(f1_per_label(Y, pred).mean()),
            "micro_f1": float(2 * tp / max(2 * tp + fp + fn, 1)),
            "subset_acc": float((pred == Y).all(1).mean()),
            "avg_pred_labels": float(pred.sum(1).mean())}


# Threshold tuning
GRID = np.round(np.arange(0.02, 0.96, 0.01), 2)


def f1_curves(P, Y):
    """(len(GRID), L) matrix: F1 of every label at every global threshold"""
    Yb = Y.astype(bool)
    out = np.zeros((len(GRID), P.shape[1]))
    for gi, t in enumerate(GRID):
        out[gi] = f1_per_label(Yb, P >= t)
    return out


def _smooth(curves, w=5):
    pad = w // 2
    c = np.pad(curves, ((pad, pad), (0, 0)), mode="edge")
    k = np.ones(w) / w
    return np.stack([np.convolve(c[:, j], k, mode="valid") for j in range(c.shape[1])], 1)


def fit_thresholds(P, Y, mode="blend", lam=0.5):
    """mode: global (one t) | perlabel (one t per label, smoothed F1 curve) | blend (lam*perlabel + (1-lam)*global)"""
    curves = f1_curves(P, Y)
    g = GRID[np.argmax(curves.mean(1))]
    t_l = GRID[np.argmax(_smooth(curves), 0)]
    if mode == "global":
        return np.full(P.shape[1], g)
    if mode == "perlabel":
        return t_l.astype(np.float64)
    return lam * t_l + (1 - lam) * g


def apply_thresholds(P, thr, force_top1=True):
    """multi-hot prediction; every document has >=1 statute in train/dev, so empty rows get their best-ratio label"""
    pred = P >= thr[None, :]
    if force_top1:
        empty = np.where(~pred.any(1))[0]
        if len(empty):
            pred[empty, np.argmax(P[empty] / thr[None, :], 1)] = True
    return pred


def crossfit_report(P, Y, modes=("global", "perlabel", "blend"), folds=2, seed=0):
    """honest dev estimate: thresholds fitted on the other half of dev, scored on this half"""
    rng = np.random.RandomState(seed)
    perm = rng.permutation(len(P))
    parts = np.array_split(perm, folds)
    res = {}
    for mode in modes:
        pred = np.zeros(P.shape, dtype=bool)
        for k in range(folds):
            te = parts[k]
            tr = np.concatenate([parts[j] for j in range(folds) if j != k])
            thr = fit_thresholds(P[tr], Y[tr], mode)
            pred[te] = apply_thresholds(P[te], thr)
        res[mode] = all_metrics(Y, pred)
    return res

In [ ]:
# Model architecture
# Keep model layer names compatible with the pretrained checkpoint.
@dataclass
class ModelConfig:
    vocab_size: int = 32000
    d_model: int = 640
    n_layers: int = 8
    n_heads: int = 10
    n_kv_heads: int = 2
    ffn_hidden: int = 2048
    dropout: float = 0.2
    max_len: int = 1088
    pad_id: int = 0
    unk_id: int = 1
    eos_id: int = 2


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight).type_as(x)


def build_rope(head_dim, max_len, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    freqs = torch.outer(torch.arange(max_len).float(), inv_freq)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    x1, x2 = x.float().chunk(2, dim=-1)
    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.type_as(x)


def repeat_kv(x, groups):
    if groups == 1:
        return x
    B, KV, T, hd = x.shape
    return x[:, :, None].expand(B, KV, groups, T, hd).reshape(B, KV * groups, T, hd)


class GQAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.h, self.kv = cfg.n_heads, cfg.n_kv_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.groups = self.h // self.kv
        self.q_proj = nn.Linear(cfg.d_model, self.h * self.hd, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.o_proj = nn.Linear(self.h * self.hd, cfg.d_model, bias=False)

    def forward(self, x, cos, sin):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        k, v = repeat_kv(k, self.groups), repeat_kv(v, self.groups)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, -1))


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.w_gate = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_up = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_down = nn.Linear(cfg.ffn_hidden, cfg.d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.attn = GQAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model)
        self.ffn = SwiGLU(cfg)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, cos, sin):
        x = x + self.drop(self.attn(self.norm1(x), cos, sin))
        x = x + self.drop(self.ffn(self.norm2(x)))
        return x


class DecoderLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = RMSNorm(cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        cos, sin = build_rope(cfg.d_model // cfg.n_heads, cfg.max_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)


def load_phase1_backbone():
    cfg = ModelConfig(**{k: v for k, v in PH1_CFG.items() if k in ModelConfig.__dataclass_fields__})
    sd = {re.sub(r"^(module\.|_orig_mod\.)+", "", k): v for k, v in PH1["model"].items()}
    lm = DecoderLM(cfg)
    lm.load_state_dict(sd, strict=True)          # fails loudly if architecture/keys differ
    assert CHUNK_LEN <= cfg.max_len, "CHUNK_LEN exceeds the RoPE table of the pretrained model"
    return lm


class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha, dropout, rs=False):
        super().__init__()
        self.base = base
        self.scale = alpha / (math.sqrt(r) if rs else r)
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))      # B = 0  ->  starts exactly at the pretrained model
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale


def inject_lora(lm):
    for p in lm.parameters():
        p.requires_grad = False
    n = 0
    for blk in lm.blocks:
        for parent in (blk.attn, blk.ffn):
            for name in LORA_TARGETS:
                if hasattr(parent, name) and isinstance(getattr(parent, name), nn.Linear):
                    setattr(parent, name, LoRALinear(getattr(parent, name), LORA_R, LORA_ALPHA, LORA_DROPOUT, USE_RSLORA))
                    n += 1
    lm.final_norm.weight.requires_grad = True      # tiny, helps adapt the pooled representation
    for m in lm.modules():                         # lighter residual dropout than in pretraining
        if isinstance(m, nn.Dropout) and m.p == lm.cfg.dropout:
            m.p = FT_DROPOUT
    return n


class LSIModel(nn.Module):
    """
    chunks of a document -> frozen+LoRA decoder -> [mean-pool ; last-token] per chunk -> MLP
    -> label-wise attention over the document's chunks (each statute learns WHICH chunks to read)
    -> one logit per statute.  Multi-label: independent sigmoid per statute.
    """

    def __init__(self, lm, n_labels, prior_logits):
        super().__init__()
        self.lm = lm
        d = lm.cfg.d_model
        self.chunk_proj = nn.Sequential(nn.LayerNorm(2 * d), nn.Dropout(HEAD_DROPOUT),
                                        nn.Linear(2 * d, HEAD_HIDDEN), nn.GELU())
        self.Q = nn.Parameter(torch.randn(n_labels, HEAD_HIDDEN) * 0.1)      # label queries for attention over chunks
        self.W = nn.Parameter(torch.randn(n_labels, HEAD_HIDDEN) * 0.02)     # per-label output weights
        self.b = nn.Parameter(torch.as_tensor(prior_logits, dtype=torch.float32))   # init = log prior odds
        self.grad_ckpt = GRAD_CKPT

    def encode_chunks(self, ids, lens):
        lm = self.lm
        N, T = ids.shape
        cos, sin = lm.rope_cos[:T][None, None], lm.rope_sin[:T][None, None]
        x = lm.drop(lm.embed(ids))
        for blk in lm.blocks:
            if self.grad_ckpt and self.training and torch.is_grad_enabled():
                x = checkpoint(blk, x, cos, sin, use_reentrant=False)
            else:
                x = blk(x, cos, sin)
        x = lm.final_norm(x).float()
        # Causal attention prevents real tokens from attending to right-side padding.
        mask = (torch.arange(T, device=ids.device)[None, :] < lens[:, None]).unsqueeze(-1)
        mean = (x * mask).sum(1) / lens[:, None].float()
        last = x[torch.arange(N, device=ids.device), lens - 1]
        return torch.cat([mean, last], dim=-1)

    def forward(self, ids, lens, owner, pos, n_docs):
        u = self.chunk_proj(self.encode_chunks(ids, lens))                    # (N, H)
        Cmax = int(pos.max().item()) + 1
        U = u.new_zeros(n_docs, Cmax, u.size(-1)).index_put((owner, pos), u)  # (D, C, H)
        valid = torch.zeros(n_docs, Cmax, dtype=torch.bool, device=u.device)
        valid[owner, pos] = True
        scores = torch.einsum("dch,lh->dcl", U, self.Q.to(U.dtype)).float() / math.sqrt(U.size(-1))
        scores = scores.masked_fill(~valid[:, :, None], float("-inf"))
        alpha = torch.softmax(scores, dim=1)                                  # attention over chunks, per statute
        doc = torch.einsum("dcl,dch->dlh", alpha.to(U.dtype), U)              # (D, L, H)
        return (doc.float() * self.W).sum(-1) + self.b                        # (D, L) logits


def build_model():
    lm = load_phase1_backbone()
    n_lora = inject_lora(lm)
    prior = np.clip(Y_train.mean(0), 1e-4, 1 - 1e-4)
    model = LSIModel(lm, N_LABELS, np.log(prior / (1 - prior))).to(DEVICE)
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_all = sum(p.numel() for p in model.parameters())
    log(f"LoRA modules: {n_lora} | trainable {n_tr / 1e6:.2f}M / total {n_all / 1e6:.1f}M params "
        f"({100 * n_tr / n_all:.1f}%)")
    return model


def trainable_state(model):
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}


def load_trainable(model, sd):
    missing, unexpected = model.load_state_dict(sd, strict=False)
    assert not unexpected, f"unexpected keys in checkpoint: {unexpected[:5]}"
    train_names = {n for n, p in model.named_parameters() if p.requires_grad}
    assert not (train_names & set(missing)), "some trainable tensors were not restored"

In [ ]:
# Batching and inference
def doc_cost(n_tokens, k):
    """approx. padded tokens a doc costs in a batch: (#chunks used) x (chunk length)"""
    return min(n_chunks_of(n_tokens), k) * min(CHUNK_LEN, n_tokens)


def pack(doc_order, store, k, budget, max_docs):
    batches, cur, cost = [], [], 0
    for d in doc_order:
        c = doc_cost(int(store.lens[d]), k)
        if cur and (cost + c > budget or len(cur) >= max_docs):
            batches.append(cur)
            cur, cost = [], 0
        cur.append(int(d))
        cost += c
    if cur:
        batches.append(cur)
    return batches


def make_train_batches(store, epoch):
    """deterministic given (SEED, epoch): shuffle -> sort by length inside pools (less padding) -> pack -> shuffle batches"""
    rng = np.random.RandomState(SEED + 1000 * epoch)
    order = rng.permutation(len(store))
    POOL = 1536
    batches = []
    for p in range(0, len(order), POOL):
        pool = sorted(order[p:p + POOL], key=lambda d: store.lens[d])
        batches += pack(pool, store, MAX_CHUNKS_TRAIN, TRAIN_TOKEN_BUDGET, MAX_DOCS_PER_BATCH)
    perm = rng.permutation(len(batches))
    return [batches[i] for i in perm]


def make_eval_batches(store):
    order = np.argsort(store.lens)
    return pack(order, store, MAX_CHUNKS_EVAL, EVAL_TOKEN_BUDGET, 128)


def collate(store, doc_ids, k, rng=None):
    seqs, owner, pos = [], [], []
    for bi, d in enumerate(doc_ids):
        a, b = int(store.offsets[d]), int(store.offsets[d + 1])
        n = b - a
        for p, s in enumerate(pick_starts(chunk_starts(n), k, rng)):
            seqs.append(store.tokens[a + s: a + min(n, s + CHUNK_LEN)])
            owner.append(bi)
            pos.append(p)
    lens = np.array([len(x) for x in seqs], dtype=np.int64)
    ids = np.full((len(seqs), int(lens.max())), PAD_ID, dtype=np.int64)
    for i, x in enumerate(seqs):
        ids[i, :len(x)] = x
    t = lambda a: torch.from_numpy(a).to(DEVICE, non_blocking=True)
    return (t(ids), t(lens), t(np.asarray(owner, dtype=np.int64)), t(np.asarray(pos, dtype=np.int64)), len(doc_ids))


@torch.no_grad()
def predict_probs(model, store, name=""):
    """sigmoid probabilities (n_docs, N_LABELS), rows aligned with the store"""
    model.eval()
    P = np.zeros((len(store), N_LABELS), dtype=np.float32)
    batches = make_eval_batches(store)
    t0 = time.time()
    for i, docs in enumerate(batches):
        ids, lens, owner, pos, nd = collate(store, docs, MAX_CHUNKS_EVAL, rng=None)
        with amp_ctx():
            logits = model(ids, lens, owner, pos, nd)
        P[docs] = torch.sigmoid(logits.float()).cpu().numpy()
        if i % 50 == 0:
            log(f"  predict {name}: {sum(len(b) for b in batches[:i + 1]):,}/{len(store):,} docs | "
                f"{(time.time() - t0) / 60:.1f} min")
    return P

In [ ]:
# Training with checkpoint resume
# Save the latest state for resume and the best model by dev macro-F1.
T_START = time.time()

# Give rare positive labels a slightly higher weight.
_pos = Y_train.sum(0).astype(np.float64)
POS_W = torch.tensor(np.clip(np.sqrt((len(Y_train) - _pos) / np.maximum(_pos, 1)), 1.0, POS_WEIGHT_CAP),
                     dtype=torch.float32, device=DEVICE)
log(f"pos_weight range: {POS_W.min().item():.2f} .. {POS_W.max().item():.2f}")

Y_train_t = torch.from_numpy(Y_train.astype(np.float32))


def make_optimizer(model):
    groups = {"lora": [], "norm": [], "head_w": [], "head_nd": []}
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if n.startswith("lm.") and "final_norm" in n:
            groups["norm"].append(p)
        elif n.startswith("lm."):
            groups["lora"].append(p)
        elif p.ndim >= 2:
            groups["head_w"].append(p)
        else:
            groups["head_nd"].append(p)
    spec = [("lora", LORA_LR, WEIGHT_DECAY), ("norm", LORA_LR, 0.0),
            ("head_w", HEAD_LR, WEIGHT_DECAY), ("head_nd", HEAD_LR, 0.0)]
    pg = [{"params": groups[k], "lr": lr, "base_lr": lr, "weight_decay": wd} for k, lr, wd in spec if groups[k]]
    return torch.optim.AdamW(pg, betas=(0.9, 0.99), eps=1e-8, fused=(DEVICE.type == "cuda"))


def lr_mult(step, total):
    if step < WARMUP_STEPS:
        return (step + 1) / WARMUP_STEPS
    prog = min(1.0, (step - WARMUP_STEPS) / max(1, total - WARMUP_STEPS))
    return MIN_LR_RATIO + (1 - MIN_LR_RATIO) * 0.5 * (1 + math.cos(math.pi * prog))


def find_previous_latest():
    best = None
    for d in walk_find("lsi_latest_meta.json", [CKPT_DIR, INPUT_ROOT]):
        meta = read_json(os.path.join(d, "lsi_latest_meta.json"))
        if not meta or meta.get("tag") != RUN_TAG or meta.get("n_labels") != N_LABELS:
            continue
        if not os.path.exists(os.path.join(d, "lsi_latest.pt")):
            continue
        if best is None or meta["global_step"] > best[1]["global_step"]:
            best = (d, meta)
    return best


def carry_over_best():
    """copy the best checkpoint of an earlier session (attached as input) into this session's checkpoint dir"""
    cur = read_json(f"{CKPT_DIR}/lsi_best_meta.json")
    for d in walk_find("lsi_best_meta.json", [INPUT_ROOT]):
        meta = read_json(os.path.join(d, "lsi_best_meta.json"))
        if not meta or meta.get("tag") != RUN_TAG or not os.path.exists(os.path.join(d, "lsi_best.pt")):
            continue
        if cur is None or meta["dev_macro_f1"] > cur["dev_macro_f1"]:
            shutil.copy(os.path.join(d, "lsi_best.pt"), f"{CKPT_DIR}/lsi_best.pt.tmp")
            os.replace(f"{CKPT_DIR}/lsi_best.pt.tmp", f"{CKPT_DIR}/lsi_best.pt")
            shutil.copy(os.path.join(d, "lsi_best_meta.json"), f"{CKPT_DIR}/lsi_best_meta.json")
            cur = meta
    return cur


def dev_eval(model):
    """dev macro-F1 with the best single global threshold (cheap, comparable across epochs)"""
    P = predict_probs(model, stores["dev"], "dev")
    thr = fit_thresholds(P, Y_dev, "global")
    m = all_metrics(Y_dev, apply_thresholds(P, thr))
    log(f"  DEV  macro-F1 {m['macro_f1']:.4f} | micro-F1 {m['micro_f1']:.4f} | subset-acc {m['subset_acc']:.4f} "
        f"| global thr {thr[0]:.2f} | avg labels/doc {m['avg_pred_labels']:.2f}")
    return m


def train(model):
    best_meta = carry_over_best()
    best_f1 = best_meta["dev_macro_f1"] if best_meta else -1.0
    opt = make_optimizer(model)
    scaler = torch.amp.GradScaler("cuda", enabled=(AMP["enabled"] and AMP["dtype"] == torch.float16))
    state = {"epoch": 0, "bi": 0, "global_step": 0}

    prev = find_previous_latest()
    if prev is not None:
        d, meta = prev
        ck = torch.load(os.path.join(d, "lsi_latest.pt"), map_location="cpu", weights_only=False)
        load_trainable(model, ck["trainable"])
        opt.load_state_dict(ck["opt"])
        scaler.load_state_dict(ck["scaler"])
        state.update(epoch=ck["epoch"], bi=ck["bi"], global_step=ck["global_step"])
        log(f"RESUMED from {d}: epoch {state['epoch'] + 1}, batch {state['bi']}, step {state['global_step']}")
        del ck

    n_batches0 = len(make_train_batches(stores["train"], 0))
    total_steps = EPOCHS * n_batches0
    log(f"{len(stores['train']):,} train docs | ~{n_batches0:,} steps/epoch | {EPOCHS} epochs | ~{total_steps:,} steps")

    def save_latest():
        atomic_torch_save({"trainable": trainable_state(model), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                           "epoch": state["epoch"], "bi": state["bi"], "global_step": state["global_step"],
                           "tag": RUN_TAG}, f"{CKPT_DIR}/lsi_latest.pt")
        atomic_json({"tag": RUN_TAG, "n_labels": N_LABELS, "epoch": state["epoch"], "bi": state["bi"],
                     "global_step": state["global_step"], "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")},
                    f"{CKPT_DIR}/lsi_latest_meta.json")

    def eval_and_maybe_save_best(tag):
        nonlocal best_f1
        m = dev_eval(model)
        if m["macro_f1"] > best_f1:
            best_f1 = m["macro_f1"]
            atomic_torch_save({"trainable": trainable_state(model), "dev_macro_f1": best_f1, "tag": RUN_TAG},
                              f"{CKPT_DIR}/lsi_best.pt")
            atomic_json({"tag": RUN_TAG, "dev_macro_f1": best_f1, "where": tag, "global_step": state["global_step"],
                         "label_ids": LABEL_IDS, "n_labels": N_LABELS}, f"{CKPT_DIR}/lsi_best_meta.json")
            log(f"  >> NEW BEST dev macro-F1 {best_f1:.4f} ({tag}) - lsi_best.pt saved")
        return m

    budget_s = (TIME_BUDGET_H - FINAL_RESERVE_H) * 3600
    timed_out = False
    t_loop = time.time()
    steps_this_run, chunks_seen, run_loss = 0, 0, 0.0

    for epoch in range(state["epoch"], EPOCHS):
        batches = make_train_batches(stores["train"], epoch)
        model.train()
        for bi in range(state["bi"], len(batches)):
            docs = batches[bi]
            rng = np.random.RandomState((SEED * 7919 + epoch * 100003 + bi) % (2 ** 32 - 1))
            ids, lens, owner, pos, nd = collate(stores["train"], docs, MAX_CHUNKS_TRAIN, rng)
            y = Y_train_t[docs].to(DEVICE)

            mult = lr_mult(state["global_step"], total_steps)
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * mult

            with amp_ctx():
                logits = model(ids, lens, owner, pos, nd)
            loss_doc = F.binary_cross_entropy_with_logits(logits.float(), y, pos_weight=POS_W, reduction="none").mean(1)
            loss = loss_doc.sum() / DOC_NORM
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"]], GRAD_CLIP)
            scaler.step(opt)
            scaler.update()

            state["global_step"] += 1
            state["epoch"], state["bi"] = epoch, bi + 1
            steps_this_run += 1
            chunks_seen += int(lens.numel())
            run_loss += loss_doc.mean().item()

            if steps_this_run % LOG_EVERY == 0:
                el = time.time() - t_loop
                eta_epoch = (len(batches) - bi - 1) * el / steps_this_run
                log(f"ep {epoch + 1}/{EPOCHS} step {bi + 1}/{len(batches)} (global {state['global_step']}) | "
                    f"loss {run_loss / LOG_EVERY:.4f} | lr x{mult:.3f} | {chunks_seen / el:.1f} chunks/s | "
                    f"epoch ETA {fmt_time(eta_epoch)} | total elapsed {fmt_time(time.time() - T_START)}")
                run_loss = 0.0
            if state["global_step"] % SAVE_EVERY == 0:
                save_latest()
            if time.time() - T_START > budget_s:
                timed_out = True
                break

        if timed_out:
            log(f"time budget reached inside epoch {epoch + 1} - saving and moving on to dev tuning / prediction")
            save_latest()
            eval_and_maybe_save_best(f"timeout@epoch{epoch + 1}")
            break
        state["epoch"], state["bi"] = epoch + 1, 0
        save_latest()
        eval_and_maybe_save_best(f"epoch{epoch + 1}")
        model.train()

    log(f"training finished | best dev macro-F1 (global thr) {best_f1:.4f} | elapsed {fmt_time(time.time() - T_START)}")
    return best_f1


model = build_model()
BEST_DEV_F1 = train(model)

In [ ]:
# Evaluate, predict, and create the submission
bm = read_json(f"{CKPT_DIR}/lsi_best_meta.json")
assert bm is not None and os.path.exists(f"{CKPT_DIR}/lsi_best.pt"), "no best checkpoint found - did training run?"
assert bm["label_ids"] == LABEL_IDS, "label space differs from the one the checkpoint was trained with"
ck = torch.load(f"{CKPT_DIR}/lsi_best.pt", map_location="cpu", weights_only=False)
load_trainable(model, ck["trainable"])
log(f"loaded best checkpoint (dev macro-F1 {ck['dev_macro_f1']:.4f}, {bm['where']})")

# Dev predictions and threshold tuning
P_dev = predict_probs(model, stores["dev"], "dev")
cf = crossfit_report(P_dev, Y_dev)          # thresholds fitted on one half of dev, scored on the other half
log("\nDEV, honest cross-fitted estimate (force >=1 label per document):")
for mode, m in cf.items():
    log(f"  {mode:9s} macro-F1 {m['macro_f1']:.4f} | micro-F1 {m['micro_f1']:.4f} | "
        f"subset-acc {m['subset_acc']:.4f} | avg labels/doc {m['avg_pred_labels']:.2f}")
BEST_MODE = max(cf, key=lambda k: cf[k]["macro_f1"])
THR = fit_thresholds(P_dev, Y_dev, BEST_MODE)          # final thresholds use all of dev
pred_dev = apply_thresholds(P_dev, THR)
m_dev = all_metrics(Y_dev, pred_dev)
log(f"chosen strategy: {BEST_MODE} | in-sample dev macro-F1 {m_dev['macro_f1']:.4f} (optimistic, as thresholds were fitted on it)")
log(f"thresholds: min {THR.min():.2f} | median {np.median(THR):.2f} | max {THR.max():.2f}")

f1_l = f1_per_label(Y_dev, pred_dev)
worst = np.argsort(f1_l)[:8]
log("weakest statutes on dev (id: F1, train positives): " +
    ", ".join(f"{LABEL_IDS[i]}: {f1_l[i]:.2f} ({int(Y_train[:, i].sum())})" for i in worst))

# Test predictions
P_test = predict_probs(model, stores["test"], "test")
pred_test = apply_thresholds(P_test, THR)

preds = []
valid_ids = set(LABEL_IDS)
for i, doc_id in enumerate(TEST_IDS):
    ids = sorted({int(LABEL_IDS[j]) for j in np.where(pred_test[i])[0]})       # unique ints, valid statute ids only
    assert ids and all(x in valid_ids for x in ids)
    preds.append({"id": doc_id, "pred": ids})

assert len(preds) == len(TEST_IDS) == len(stores["test"])
assert len({str(p["id"]) for p in preds}) == len(preds), "duplicate test ids"
log(f"test: {len(preds):,} predictions | avg statutes/doc {np.mean([len(p['pred']) for p in preds]):.2f} "
    f"(dev truth: {Y_dev.sum(1).mean():.2f})")

# Save outputs
pred_path = f"{OUT_DIR}/prediction.json"
with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(preds, f, ensure_ascii=False)
zip_path = f"{OUT_DIR}/submission.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(pred_path, arcname="prediction.json")            # prediction.json at the zip root

np.save(f"{OUT_DIR}/dev_probs.npy", P_dev)                   # keep for ensembling with the other Phase-1 models
np.save(f"{OUT_DIR}/test_probs.npy", P_test)
atomic_json({"label_ids": LABEL_IDS, "thresholds": [float(x) for x in THR], "strategy": BEST_MODE,
             "dev_crossfit": cf, "dev_in_sample": m_dev, "test_ids": [str(x) for x in TEST_IDS]},
            f"{OUT_DIR}/thresholds_and_report.json")
log("saved:", pred_path, "|", zip_path, "| dev_probs.npy | test_probs.npy | thresholds_and_report.json")
log("done.")